# Figure: Shorkie-LM architecture schematic

This figure draws the architecture of the Shorkie masked-DNA language model (the `unet_small_bert_drop` configuration). The model takes a 16,384 bp one-hot DNA window and passes it through a DNA convolution stem, a residual convolution tower (7 downsampling blocks), 8 self-attention transformer blocks, and a 7-step U-Net upsampling decoder, ending in a per-nucleotide 4-way softmax head that predicts the masked bases (A/C/G/T). The schematic is rendered as a vertical block diagram, with each block annotated by its `params.json` hyperparameters.

**Reproduces:** the Shorkie-LM architecture diagram (model schematic).

**Upstream:** Runs end-to-end from released data — it reads only the committed/released `params.json` of the language model; no upstream `scripts/` stage is required. The original pipeline built the full Keras graph and called `tensorflow.keras.utils.plot_model` (see Source script), which requires a GPU/TensorFlow model build; this notebook instead renders the same architecture faithfully and CPU-only directly from the `params.json` `model` block.

**Requires:** the `yeast_ml` conda env with the package installed (`pip install -e .`). CPU-only — no GPU, no TensorFlow, no baskerville model build needed.

**Source script:** `scripts/04_analysis/others/viz_shorkie_lm_arch/viz_lm.py` (which invokes baskerville's `hound_model_viz.py` -> `plot_model`).

In [ ]:
import json

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

from shorkie import config

In [ ]:
# Resolve the released LM params.json via the package config (no hardcoded paths).
params_file = config.path("models.shorkie_lm") / "params.json"
print("Reading architecture from:", params_file)

with open(params_file) as fh:
    params = json.load(fh)

params_model = params["model"]
print("Top-level model keys:", list(params_model.keys()))
print("seq_length :", params_model.get("seq_length"))
print("activation :", params_model.get("activation"))
print("norm_type  :", params_model.get("norm_type"))

In [ ]:
# Flatten the trunk (+ head) into an ordered list of display blocks.
# Each entry: (title, list-of-detail-lines, color-group). `repeat`/`res_tower`
# blocks are shown as a single stacked block annotated with their repeat count,
# mirroring how the architecture is conceptually grouped.
seq_length = params_model.get("seq_length", 16384)

# input block
blocks = [("Input DNA one-hot", [f"({seq_length:,}, 4)  A/C/G/T"], "input")]

def _fmt(d, keys):
    out = []
    for k in keys:
        if k in d and d[k] is not None:
            out.append(f"{k}={d[k]}")
    return out

for layer in params_model["trunk"]:
    name = layer["name"]
    if name == "conv_dna":
        det = _fmt(layer, ["filters", "kernel_size", "activation"])
        blocks.append(("conv_dna  (DNA stem)", det, "conv"))
    elif name == "res_tower":
        rep = layer.get("repeat", 1)
        det = _fmt(layer, ["filters_init", "filters_end", "kernel_size",
                           "num_convs", "pool_size", "dropout"])
        blocks.append((f"res_tower  (x{rep} downsample)", det, "res"))
    elif name == "transformer_tower":
        rep = layer.get("repeat", 1)
        det = _fmt(layer, ["heads", "key_size", "num_position_features", "dropout"])
        blocks.append((f"transformer_tower  (x{rep} MHA)", det, "attn"))
    elif name == "unet_conv":
        # collapse the consecutive U-Net upsampling blocks into one stacked block
        if blocks and blocks[-1][0].startswith("unet_conv"):
            n = blocks[-1][2]  # reuse count stash
            title = f"unet_conv  (x{n + 1} upsample)"
            det = _fmt(layer, ["kernel_size", "upsample_conv"])
            blocks[-1] = (title, det, n + 1)
        else:
            det = _fmt(layer, ["kernel_size", "upsample_conv"])
            blocks.append(("unet_conv  (x1 upsample)", det, 1))
    else:
        blocks.append((name, _fmt(layer, list(layer.keys())[:4]), "other"))

# normalize the unet stash (count int -> color group string)
blocks = [(t, d, ("unet" if isinstance(c, int) else c)) for (t, d, c) in blocks]

# output head
head = params_model.get("head_human", {})
hdet = _fmt(head, ["units", "activation"])
blocks.append(("final head  (per-nt logits)", hdet or ["units=4", "activation=softmax"], "head"))

for t, d, g in blocks:
    print(f"{t:36s} {d}")

In [ ]:
# Render the vertical block diagram.
colors = {
    "input": "#d9d9d9",
    "conv":  "#a6cee3",
    "res":   "#1f78b4",
    "attn":  "#fb9a99",
    "unet":  "#33a02c",
    "head":  "#ff7f00",
    "other": "#cccccc",
}

n = len(blocks)
box_w, box_h, gap = 6.0, 1.0, 0.7
fig_h = max(7, n * (box_h + gap))
fig, ax = plt.subplots(figsize=(7.5, fig_h))

y = (n - 1) * (box_h + gap)
centers = []
for (title, det, group) in blocks:
    fc = colors.get(group, "#cccccc")
    rect = mpatches.FancyBboxPatch(
        (0, y), box_w, box_h,
        boxstyle="round,pad=0.02,rounding_size=0.12",
        linewidth=1.2, edgecolor="black", facecolor=fc, alpha=0.92,
    )
    ax.add_patch(rect)
    txt_col = "white" if group in ("res", "unet", "head") else "black"
    ax.text(box_w / 2, y + box_h * 0.62, title, ha="center", va="center",
            fontsize=10, fontweight="bold", color=txt_col)
    if det:
        ax.text(box_w / 2, y + box_h * 0.22, "  ·  ".join(det), ha="center",
                va="center", fontsize=7.5, color=txt_col)
    centers.append(y + box_h / 2)
    y -= (box_h + gap)

# downward arrows between consecutive blocks
for i in range(n - 1):
    top = centers[i] - box_h / 2
    bot = centers[i + 1] + box_h / 2
    ax.add_patch(FancyArrowPatch(
        (box_w / 2, top), (box_w / 2, bot),
        arrowstyle="-|>", mutation_scale=14, linewidth=1.1, color="black",
    ))

ax.set_xlim(-0.5, box_w + 0.5)
ax.set_ylim(-0.5, n * (box_h + gap))
ax.axis("off")
ax.set_title("Shorkie-LM  —  unet_small_bert_drop", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

The diagram is read top-to-bottom: a 16,384 bp one-hot DNA window enters the `conv_dna` stem, is contracted by the residual convolution tower, contextualized by 8 multi-head self-attention transformer blocks, then expanded back to single-nucleotide resolution by the 7-step U-Net decoder. The `final` head emits a 4-way softmax over A/C/G/T at every position, which the masked-language objective scores against the held-out (masked) bases. All hyperparameters annotated on each block are read verbatim from the released `params.json`, so this schematic stays in sync with the published Shorkie-LM checkpoint.